# Lab 2 - Exploratory Data Analysis (EDA)

Here, we will approach fundamental concepts of EDA to work with public datasets:

1. Gathering data
1. Data processing
1. Understand the data numerically
1. Understand the data graphically

---

```
Author:   Luis Quintero
Contact:  luis.quintero@dsv.su.se | luisqtr.com
```


In [ ]:
# First, let's import the basic libraries
#  we can still import more libraries later on...
import pandas as pd

#### Setting goals

There are **thousands** of ways in which we can process a dataset. Defining goals help us craft the analysis pipeline, and know whether we just need descriptive and diagnostic tools, or if predictive and prescriptive algorithms are needed.

A clear aim leads the analysis, but iteration and experimentation is needed.

In this case, we want:

- Load the structured dataset correctly.
- Create a processed dataset with a subset of features meaningful for our the  goal.
- Answer simple analytical questions that require filtering.
- Know which features are correlated among themselves, and with respect to the target variable.

# 1. Gathering data

First, we need to collect data related to our domain.

There are multiple ways to obtain the data. For instance, they can exist as databases in a company (e.g., `SQL`, `MongoDB`), available in an online repository accessible via API (i.e., requests to an `http` server), or as local files in text format (e.g., `.csv`, `.json`, `.txt`). In real-life is common to connect to APIs, but most processes end up in a structured dataset in text files.

All the data from databases and online repositories can be stored as files. So, **this is the most common data source**.

There are thousands of datasets available online to apply the data science framework and build machine learning models. Most of the cases, they provide one or multiple text files that we need to load into the computer's memory to be able to manipulate them.

## Example: Bank Marketing

In the rest of this notebook, we analyze the *Bank Marketing* dataset available in the UCI repository:

https://archive-beta.ics.uci.edu/dataset/222/bank+marketing

**Description**

Download the file `bank.zip` [from the repository](https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip), extract it, and open it in your computer with your preferred software (e.g., Excel, notepad).

The first step in the data science pipeline is to visually inspect the structure of the dataset. It helps us understand the problem domain, available variables, number of features, and perhaps their quality. In this case, the dataset is contained in a `.zip` with multiple files: the dataset description in `bank-names.csv`, the complete dataset `bank-full.csv` contains around 45.000 observations, and a simplified version with 10% of the data `bank.csv`

- The proposed purpose of the dataset is for a marketing campaign in a Portuguese bank, to predict whether a client would subscribe or not to a financial service (term deposit).
- The dataset contains 16 features.
- Some **numerical** variables are the client's `age` in years, `balance` in euros, the last contact `day` of the month, the `duration` of the last contact in seconds, the number of contacts to the same client in this `campaign`.
- Some **categorical** variables are the `marital` status of the client, their level of `education`, the `contact` communication type.
- Some **binary** variables are whether the client already has a credit in `default`, if the person has a `housing` loan, if they have a `personal` loan.
- The target variable `y` is also a binary variable indicating whether the client subscribes (`y='yes'`) or not (`y=no`) to the financial service

Let's use the small version, since it is sufficient for an EDA. When we have tested our pipeline in the small data, it should also work for `bank-full.csv` because the data variables are the same.

## Loading the data

Now, we are ready to start coding the pipeline described in the initial set of goals.

---

If you are working on Google Colab, the `.csv` file should be uploaded to the server's files on the left part of the screen. If you are running jupyter notebooks locally, you need to have the `.csv` file in the same folder than the jupyter notebook `.ipynb`.

In [ ]:
# Pandas provides built-in functions to load text files into DataFrames.
df = pd.read_csv("./Data/bank.csv")
df.head()

After we load the data, note that the DataFrame above does **NOT** look right!
All the values are comprised in a single column, but we want to see each column separately with its corresponding values.

The solution is to use the function `read_csv()` with parameters that describe the specific characteristics of our dataset. Check the documentation of [`pandas.read_csv()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html) and note that the default separator is a comma (the parameter `sep` says `default  ,`).

In our dataset, the columns are separated by semicolon `;`, therefore we need to override this parameter. Another common problem is arises if your dataset does not have a `header`, or you want to load certain numbers of rows `nrows` in the file.

Reading the text files properly could save a lot of time in preprocessing. So, read the documentation thoroughly, and most likely one of the available parameters will fit the structure of your files. In this case, we will leverage other two parameters (`true_values` and `false_values`) to let pandas translate the binary target variables `y`,`default`, and `loan` from `[yes,no]` to the boolean option `[True,False]`.

In [ ]:
df = pd.read_csv("./Data/bank.csv", sep=";", true_values =["yes"], false_values=["no"])
df

In [ ]:
# What is the total size of the dataset?
df.shape

In [ ]:
# Which are the variable types?
df.dtypes

# 2. Data processing

## Selection/Slicing
We need ways to access specific parts of the dataset. Pandas calls this process **slicing** and it is one of the most crucial parts of data analysis.

An extensive documentation on slicing is available [in the pandas documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#slicing-with-labels).

The main two ways are:
- Selection by labels: Uses the column names and row names to access data
- Selection by position: Uses integers to identify a range of values

In [ ]:
# Select all rows in the specific column `age
df["age"]   # Returns a pandas.Series()

In [ ]:
# Select all columns from the specific row with index `4`
df.loc[4]   # `loc[]` iterates over valid values of the index

In [ ]:
# Select a range of rows from 5 until 10 from the column `job`
df["job"].loc[5:10]

All the examples above are selection by *label*, the example belows creates a selection by *position*. Note several elements:

- It uses the function `iloc[]`, not `loc[]` as above.
- The semicolon `:` is used select an interval `[from:until]`. The specific case `[3:]` means `from position number 3 until the last object`, whereas `[:3]` means `from beginning until position 3`, and `[:]` means `select everything`
- There are two parameters in `iloc` indicating a selection over `[rows,columns]`

In [ ]:
# Select the last 3 columns of the dataframe, between the rows 12 and 15
df.iloc[12:16,-3:]  # Slicing with semicolon, does not include the upper limit (16)

In [ ]:
# Select again 'by label' the rows 20 to 27, in both columns `age` and `y`
df.loc[ 12:16, ["previous","poutcome","y"]]

## Assigning correct data types

Note above that some of the variables in the dataset as detected as type `object`, meaning that their type is not identified.

We can explicitly assign data types to the columns. In this case, assigning the *categorical* variables as such.

Note that the variable `month` is requires encoding as an *ordinal categorical* variable because the order matters.

In [ ]:
# Explicitly convert the nominal categorical variables
df["job"] = df["job"].astype("category")
df["marital"] = df["marital"].astype("category")
df["education"] = df["education"].astype("category")
df["contact"] = df["contact"].astype("category")
df["poutcome"] = df["poutcome"].astype("category")

In [ ]:
# Explicitly convert the ordinal categorical variables

# 1. Create a list with the desired order
categ = ['jan', 'feb', 'mar', 'apr',
         'may', 'jun', 'jul', 'aug',
         'sep', 'oct', 'nov', 'dec']
# 2. Assign the ordinal category
df["month"] = pd.Categorical(df["month"], categories=categ, ordered=True)

# Check the types
df.dtypes

## Renaming/Deleting columns

Rename some column names to make them more descriptive. Note that the parameter `columns` expects a `dictionary` that contains the *key:value* pair corresponding to `old_name:new_name` for each column name to modify.



In [ ]:
# Replace the column names and store it in the same variable name
df = df.rename(columns={
                          "housing":"has_housing_loan",
                          "loan": "has_personal_loan",
                          })

# Assume we don't want to use the variable `default` anymore
# The parameter `inplace` means that the original dataframe is replaced
df.drop("default", axis=1, inplace=True)

# Show three random samples of the data
df.sample(3)

## Filtering with conditions

In this example, select people older than 30, married, and who subscribed to the financial service.

All the examples above where just **visualizing** a subset of the data, but it does not modify the original dataframe. To store a copy of the dataframe in memory it is needed to declare the variable explicitly, as it is shown below with `df2`.

In [ ]:
# Filter and store the result in a different dataframe `df2`
df_filtered = df[ (df["age"]>30) & (df["marital"]=="married") & (df["y"]==True) ]
df_filtered

## Create a file with processed data

After preparing the data, it is useful to store a copy of the preprocessed dataset in the disk. Specially if the original datasets are big and we have already done the selection of the variables that we would like to analyze.

Next, we filter some columns and create a copy of the dataframe in a file `bank_processed.csv`.

In [ ]:
# Deleting irrelevant features
columns_to_delete = ["job", "education", "contact", "day", "month",
                      "campaign", "pdays", "previous", "poutcome"]

# Delete the columns and assign a new variable `df_processed`
#     axis=1 means that the operation is executed in the columns
#     axis=0 is in the rows
df_processed = df.drop(columns_to_delete, axis=1)
df_processed

In [ ]:
# Save the processed dataframe in `./bank_processed.csv`
df_processed.to_csv("./Data/bank_processed.csv",
                    index=False) # We do not store the index (row number) because it isn't useful information

# 3. Exploratory Data Analysis (EDA)

Assume we have already filtered the initial dataset to the variables that we consider relevant. Let's work with `bank_processed.csv`.

In [ ]:
# Load our new processed dataset from disk
data = pd.read_csv("./Data/bank_processed.csv")

# Show the first rows
data.head()

## Understanding the data numerically

### Descriptive statistics

In [ ]:
# Describes the numerical variables
data.describe()

In [ ]:
# Categorical variables can described in isolation
data[ ["marital","has_housing_loan"] ].describe()

In [ ]:
# Describe all variables independent of their type
data.describe(include='all')

### Missing values

Features in the dataset often contain missing values for diverse reasons (lack of access to the data, errors inputing data, etc.). Missing values in the dataset are often shown as `NA`, `NaN`, `?`, `0`, or similar depending on the specific dataset.

There are many techniques to deal with missing values. [Read more here](https://pandas.pydata.org/pandas-docs/stable/user_guide/missing_data.html).

When we have a lot of data samples, it is common just to delete the rows with missing values.

In [ ]:
# Count missing values per feature
data.isnull().sum()

In [ ]:
# This dataset does not contain missing values.

# But if it had any, you could delete the rows using:
data.dropna(inplace=True)

When your dataset is small or each data sample has been carefully collected.

It is not convenient to delete the samples, you can instead impute (replace) missing values.

*The Bank dataset doesn't have missing values so these lines have no effect*

In [ ]:
# For a continuous numerical value it is common to use the mean().
imputed_value_cont_num = data["balance"].mean()
data["balance"] = data["balance"].fillna(imputed_value_cont_num)

# For discrete numerical values, it is common to use the median()
imputed_value_disc_num = data["age"].median()
data["age"] = data["age"].fillna(imputed_value_disc_num)

# For categorical values, it is common to use the mode()
imputed_value_cat = data["marital"].mode()
data["marital"] = data["marital"].fillna(imputed_value_cat)

In [ ]:
print("The value used for imputation in `balance` was \t", imputed_value_cont_num)
print("The value used for imputation in `age` was \t", imputed_value_disc_num)
print("The value used for imputation in `marital` was \t", imputed_value_cat.values)

### Answering simple questions with data

**Filtering data rows:** The basic question is to select users with determined conditions and execute summary functions on them.

**Q:** *What is the percentage of subscribers among the customers with a balance greater than 10.000 euros and without housing loan?*

In [ ]:
data_filtered = data[ (data["balance"]>10000) & (data["has_housing_loan"]==False) ]
data_filtered

In [ ]:
# Percentage of subscribers in the filtered group
number_subscribers = data_filtered[ data_filtered["y"]==True ].shape[0]
perc_subscribers = (number_subscribers / data_filtered.shape[0]) * 100

print("The total number of subscribers in the subgroup was", perc_subscribers,
      "%, equivalent to", number_subscribers, "people")

**Grouping:** Groupby is an important function to summarize multiple variables. [Read the documentation here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html). It resembles a pivot table in Excel.

**Q:** *What is the number of customers per each marital status?*

In [ ]:
data.groupby(by="marital").size()

**Q:** *What is the average balance (in euros) depending on marital status and whether they have a personal loan?*

In [ ]:
data.groupby(by=["marital", "has_personal_loan"])["balance"].mean()

In [ ]:
# Statistical moments of numeric variables
print(f'Average of balance: { data["balance"].mean() }')
print(f'Standard deviation of balance: { data["balance"].std() }')
print(f'Median of balance: { data["balance"].median() }')

### Feature engineering

You can create variables in the same dataframe.

Here, assume we want to add a new variable `balance_sek` that represents the `balance` in Swedish kronor instead of euros.

In [ ]:
exchange_rate_eur_to_sek = 10.8
data["balance_sek"] = data["balance"] * exchange_rate_eur_to_sek
data.head()

## Understanding the data graphically

Data visualization is a powerful mechanism to understand how the description of a feature or the relationships among variables.

We commonly use `matplotlib`, but there are other libraries such as `seaborn` that provide high-level data visualization. Even `pandas` includes [simple visualization directly from the dataframes](https://pandas.pydata.org/pandas-docs/stable/user_guide/visualization.html#other-plots).

In [ ]:
# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

### Histograms

In [ ]:
# Histogram
data["age"].hist(bins=20)

## The most representative group is the age group between 30 and 35

In [ ]:
# What is the average balance (in sek) depending on marital status
# and whether they have a personal loan?

avg_balance_marital_personal = data.groupby(by=["marital", "has_personal_loan"])["balance_sek"].mean()
avg_balance_marital_personal

In [ ]:
# Unwrap the pivot table into a regular pandas dataframe
df_avg_balance_marital_personal = avg_balance_marital_personal.unstack()
df_avg_balance_marital_personal

In [ ]:
# Plot the dataframe
df_avg_balance_marital_personal.plot.bar(stacked=False)
plt.title("Average balance (in sek) per marital status and personal loan")
plt.show()

### Box plots

In [ ]:
# Useful to detect outliers in the dataset
plt.boxplot(data["age"])
plt.show()

### Scatter plot

In [ ]:
# Probability density function between age and marital status
sns.relplot(data=data, x="age", y="balance", hue="marital", size="has_housing_loan",
            sizes=(10, 100), alpha=.5, palette="muted")

### Correlations

In [ ]:
correlations = data.corr(method="pearson", numeric_only=True)
correlations

In [ ]:
data.columns

In [ ]:
# Plot the correlations visually
fig, ax = plt.subplots(figsize=(10,10))

# Defininig subplots is more flexible and allows
# deeper customization in matplotlib
cax = ax.matshow(correlations, vmin=-1, vmax=1)
fig.colorbar(cax)
axes_labels = data.columns.drop("marital") # categorical values don't have correlation
plt.xticks(list(range(axes_labels.size)), axes_labels, rotation=90)
plt.yticks(list(range(axes_labels.size)), axes_labels)
plt.show()

### Correlation analysis

Look at the correlation plot above, and reflect:

- Why are the variables `balance` and `balance_sek` presenting a high correlation? What does it mean? Was there somewhere in this code where we manually induced this strong correlation?
- Can you draw any interpretation from the relationships of other variables?
- Are there variables that are strongly correlated with the outcome variable of interest `y`? If any, what could be a possible hypothesis for the correlation of these features with the outcome?

---

# Activity - [Not graded]

1. Use Python to answer the analytical questions that you proposed for the Bank data in Lab 1 and compare with the results achieved in Excel
2. Explore the package `YData Profiling`<https://docs.profiling.ydata.ai/>, apply it to this dataset, and explore if you can answer your analytical questions from the comprehensive output.

## 1. Answer below your analytical questions

In [ ]:
# Load our new processed dataset from disk
data = pd.read_csv("./Data/bank_processed.csv")

# Show the first rows
data.head()

2. Explore YData Profiling

Check in the documentation how to install the package and how to get started with a simple visualization.

Link: https://docs.profiling.ydata.ai/latest/